# Fraud Detection Pipeline

## SETUP (Imports & Configuration)

In [1]:
import os
import gc
import time
import json
import warnings
from collections import deque

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, matthews_corrcoef,
    precision_recall_curve, roc_curve, auc
)

import xgboost as xgb
import shap

warnings.filterwarnings("ignore")

##  (CONFIGURATIONS )PARAMETERS

In [2]:
DATA_DIR = "./"
OUTPUT_DIR = "./outputs"
SAMPLE_SIZE = None
RANDOM_STATE = 42
BATCH_SIZE = 50_000
SEQUENCE_LENGTH = 10
LOOKAHEAD_K = 10
PAGERANK_EVERY_N_BATCHES = 3
FEATURE_DROP_FRACTION = 0.35
SHAP_RANKING_SAMPLE_SIZE = 5000
CV_N_SPLITS = 3
DEVICE_RECENT_WINDOW = 20
ENABLE_EXPENSIVE_CENTRALITY = False
BETWEENNESS_SAMPLE_K = 200
TUNING_SAMPLE_SIZE = 10_000
HYPERPARAM_SEARCH_ITER = 5
BETA_PRIOR_ALPHA = 1
BETA_PRIOR_BETA = 27

## Detect Xgb Device

In [4]:
def _detect_xgb_device():
    try:
        _test_model = xgb.XGBClassifier(tree_method="hist", device="cuda", n_estimators=2)
        _test_model.fit(np.zeros((10, 2), dtype=np.float32), np.array([0, 1] * 5))
        return "cuda"
    except Exception:
        return "cpu"

## CONFIG  XGB_DEVICE

In [5]:
XGB_DEVICE = _detect_xgb_device()
print(f"[startup] XGBoost device auto-detected: {XGB_DEVICE.upper()}"
      + ("" if XGB_DEVICE == "cuda" else " (no GPU/CUDA build found - training will be slower on CPU)"))
os.makedirs(OUTPUT_DIR, exist_ok=True)

[startup] XGBoost device auto-detected: CUDA


## CONFIG  Log

In [6]:
def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

 LOAD DATA

In [7]:
def load_data():
    log("Loading transaction and identity tables...")
    trans_path = os.path.join(DATA_DIR, "train_transaction.csv")
    ident_path = os.path.join(DATA_DIR, "train_identity.csv")

    if not os.path.exists(trans_path) or not os.path.exists(ident_path):
        raise FileNotFoundError(
            f"Could not find the CSV files in {DATA_DIR}.\n"
            "Download them from the Kaggle link at the top of this script, "
            "and update DATA_DIR to point to that folder."
        )

    transaction = pd.read_csv(trans_path)
    identity = pd.read_csv(ident_path)

    for tbl in (transaction, identity):
        for col in tbl.select_dtypes(include=["float64"]).columns:
            tbl[col] = tbl[col].astype(np.float32)
        for col in tbl.select_dtypes(include=["int64"]).columns:
            tbl[col] = tbl[col].astype(np.int32)

    df = transaction.merge(identity, on="TransactionID", how="left")
    del transaction, identity
    gc.collect()

    if SAMPLE_SIZE is not None:
        df = df.sort_values("TransactionDT").head(SAMPLE_SIZE).reset_index(drop=True)

    df = df.sort_values("TransactionDT").reset_index(drop=True)

    log(f"Loaded {len(df):,} rows and {df.shape[1]} columns, sorted chronologically.")
    return df

## LAYER 1: TEMPORAL BEHAVIORAL FEATURES 

In [8]:
TEMPORAL_COLS = [
    "time_since_last_txn", "amt_rolling_mean_3", "amt_drift", "txn_count_so_far",
    "txn_velocity_24h", "amt_volatility_5", "amt_trend", "amt_rolling_mean_10",
    "txns_last_1h", "txns_last_7d", "amt_shock_ratio",
    "device_switch_rate_5", "channel_switch_rate_5", "time_since_last_fraud",
]

## Add Temporal Features

In [9]:
def add_temporal_features(df):
    log("Building Layer 1: expanded temporal features (point-in-time)...")
    grouped = df.groupby("card1")

    df["time_since_last_txn"] = grouped["TransactionDT"].diff().fillna(-1)

    df["amt_rolling_mean_3"] = grouped["TransactionAmt"].transform(
        lambda s: s.shift().rolling(window=3, min_periods=1).mean()
    )
    df["amt_rolling_mean_3"] = df["amt_rolling_mean_3"].fillna(df["TransactionAmt"])
    df["amt_drift"] = df["TransactionAmt"] - df["amt_rolling_mean_3"]

    df["txn_count_so_far"] = grouped.cumcount()

    def velocity(sub):
        s = sub.set_index(pd.to_timedelta(sub["TransactionDT"], unit="s"))
        counts = s["TransactionDT"].shift(1).rolling("24h").count()
        return counts.values
    df["txn_velocity_24h"] = grouped.apply(
        lambda g: pd.Series(velocity(g), index=g.index)
    ).reset_index(level=0, drop=True).fillna(0)

    df["amt_volatility_5"] = grouped["TransactionAmt"].transform(
        lambda s: s.shift().rolling(window=5, min_periods=2).std()
    ).fillna(0)

    short_mean = grouped["TransactionAmt"].transform(
        lambda s: s.shift().rolling(window=3, min_periods=1).mean()
    )
    long_mean = grouped["TransactionAmt"].transform(
        lambda s: s.shift().rolling(window=10, min_periods=1).mean()
    )
    df["amt_trend"] = (short_mean - long_mean).fillna(0)

    df["amt_rolling_mean_10"] = long_mean.fillna(df["TransactionAmt"])

    def rolling_txn_count(sub, window):
        s = sub.set_index(pd.to_timedelta(sub["TransactionDT"], unit="s"))
        counts = s["TransactionDT"].shift(1).rolling(window).count()
        return counts.values

    df["txns_last_1h"] = grouped.apply(
        lambda g: pd.Series(rolling_txn_count(g, "1h"), index=g.index)
    ).reset_index(level=0, drop=True).fillna(0)
    df["txns_last_7d"] = grouped.apply(
        lambda g: pd.Series(rolling_txn_count(g, "7d"), index=g.index)
    ).reset_index(level=0, drop=True).fillna(0)

    mean_5 = grouped["TransactionAmt"].transform(
        lambda s: s.shift().rolling(window=5, min_periods=1).mean()
    )
    mean_5_filled = mean_5.fillna(df["TransactionAmt"])
    df["amt_shock_ratio"] = df["TransactionAmt"] / (mean_5_filled + 1e-3)

    device_filled = df["DeviceInfo"].fillna("missing_device")
    device_switch_flag = df.groupby("card1").apply(
        lambda g: (device_filled.loc[g.index] != device_filled.loc[g.index].shift(1)).astype(int)
    ).reset_index(level=0, drop=True)
    df["device_switch_rate_5"] = device_switch_flag.groupby(df["card1"]).transform(
        lambda s: s.shift().rolling(window=5, min_periods=1).mean()
    ).fillna(0)

    if "P_emaildomain" in df.columns:
        channel_filled = df["P_emaildomain"].fillna("missing_channel")
        channel_switch_flag = df.groupby("card1").apply(
            lambda g: (channel_filled.loc[g.index] != channel_filled.loc[g.index].shift(1)).astype(int)
        ).reset_index(level=0, drop=True)
        df["channel_switch_rate_5"] = channel_switch_flag.groupby(df["card1"]).transform(
            lambda s: s.shift().rolling(window=5, min_periods=1).mean()
        ).fillna(0)
    else:
        df["channel_switch_rate_5"] = 0.0

    fraud_time = df["TransactionDT"].where(df["isFraud"] == 1)
    last_fraud_time = grouped.apply(
        lambda g: fraud_time.loc[g.index].shift(1).ffill()
    ).reset_index(level=0, drop=True)
    df["time_since_last_fraud"] = (df["TransactionDT"] - last_fraud_time).fillna(-1)

    return df

## DYNAMIC SOCIAL GRAPH

In [10]:
GRAPH_COLS = [
    "graph_degree", "graph_component_size", "graph_clustering",
    "graph_pagerank", "graph_community_size", "graph_neighbor_fraud_rate",
    "graph_neighbor_fraud_count", "graph_betweenness", "graph_eigenvector",
    "graph_avg_neighbor_risk", "graph_devices_per_card",
]

## LAYER 2: DYNAMIC SOCIAL GRAPH 

In [11]:
def add_dynamic_graph_features(df):
    log("Building Layer 2: dynamic social graph features (batched)...")

    missing_mask = df["DeviceInfo"].isna()
    df.loc[missing_mask, "DeviceInfo"] = "missing_" + df.loc[missing_mask].index.astype(str)
    df["_node_id"] = df.index

    extra_edge_keys = []
    for col in ["P_emaildomain", "R_emaildomain", "id_31"]:
        if col in df.columns:
            df[col] = df[col].fillna(f"missing_{col}")
            extra_edge_keys.append(col)

    for col in ["card4", "card6"]:
        if col in df.columns:
            df[col] = df[col].fillna(f"missing_{col}")
            extra_edge_keys.append(col)

    n = len(df)
    G = nx.Graph()

    graph_degree = np.zeros(n, dtype=np.float32)
    graph_component_size = np.ones(n, dtype=np.float32)
    graph_clustering = np.zeros(n, dtype=np.float32)
    graph_pagerank = np.zeros(n, dtype=np.float32)
    graph_community_size = np.ones(n, dtype=np.float32)
    graph_neighbor_fraud_rate = np.zeros(n, dtype=np.float32)
    graph_neighbor_fraud_count = np.zeros(n, dtype=np.float32)
    graph_betweenness = np.zeros(n, dtype=np.float32)
    graph_eigenvector = np.zeros(n, dtype=np.float32)
    graph_avg_neighbor_risk = np.zeros(n, dtype=np.float32)
    graph_devices_per_card = np.ones(n, dtype=np.float32)

    node_is_fraud = {}
    node_running_risk = {}
    card_fraud_count = {}
    card_txn_count = {}
    card_devices_seen = {}
    pagerank_cache = {}
    community_cache = {}
    betweenness_cache = {}
    eigenvector_cache = {}

    card_device_index = {}
    addr_device_index = {}
    extra_indexes = {col: {} for col in extra_edge_keys}

    num_batches = int(np.ceil(n / BATCH_SIZE))

    for batch_idx in range(num_batches):
        start = batch_idx * BATCH_SIZE
        end = min(start + BATCH_SIZE, n)
        batch = df.iloc[start:end]

        degree_dict, component_size, clustering_dict = {}, {}, {}
        if G.number_of_nodes() > 0:
            degree_dict = dict(G.degree())
            for comp in nx.connected_components(G):
                size = len(comp)
                for node in comp:
                    component_size[node] = size

            MAX_DEGREE_FOR_CLUSTERING = 500
            low_degree_nodes = [nd for nd, d in degree_dict.items() if d <= MAX_DEGREE_FOR_CLUSTERING]
            clustering_dict = nx.clustering(G, nodes=low_degree_nodes) if low_degree_nodes else {}

            if batch_idx % PAGERANK_EVERY_N_BATCHES == 0 and G.number_of_edges() > 0:
                try:
                    pagerank_cache = nx.pagerank(G, max_iter=50)
                except nx.PowerIterationFailedConvergence:
                    pass
                try:
                    communities = list(nx.algorithms.community.asyn_lpa_communities(G, seed=RANDOM_STATE))
                    community_cache = {}
                    for comm in communities:
                        size = len(comm)
                        for node in comm:
                            community_cache[node] = size
                except Exception:
                    pass

                if ENABLE_EXPENSIVE_CENTRALITY:
                    try:
                        k = min(BETWEENNESS_SAMPLE_K, G.number_of_nodes())
                        betweenness_cache = nx.betweenness_centrality(G, k=k, seed=RANDOM_STATE)
                    except Exception:
                        betweenness_cache = {}
                    try:
                        eigenvector_cache = nx.eigenvector_centrality(G, max_iter=100, tol=1e-4)
                    except (nx.PowerIterationFailedConvergence, Exception):
                        eigenvector_cache = {}

        for node, card, dev, addr in zip(
            batch["_node_id"].values, batch["card1"].values,
            batch["DeviceInfo"].values, batch["addr1"].values,
        ):
            neighbor_ids = set(card_device_index.get((card, dev), ())) | set(addr_device_index.get((addr, dev), ()))
            for col in extra_edge_keys:
                val = batch.at[node, col] if node in batch.index else None
                if val is not None:
                    neighbor_ids |= set(extra_indexes[col].get((val, dev), ()))

            graph_degree[node] = len(neighbor_ids)
            if neighbor_ids:
                comp_vals = [component_size.get(nb, 1) for nb in neighbor_ids]
                graph_component_size[node] = max(comp_vals) if comp_vals else 1

                clust_vals = [clustering_dict[nb] for nb in neighbor_ids if nb in clustering_dict]
                graph_clustering[node] = float(np.mean(clust_vals)) if clust_vals else 0.0

                pr_vals = [pagerank_cache[nb] for nb in neighbor_ids if nb in pagerank_cache]
                graph_pagerank[node] = float(np.mean(pr_vals)) if pr_vals else 0.0

                comm_vals = [community_cache.get(nb, 1) for nb in neighbor_ids]
                graph_community_size[node] = max(comm_vals) if comm_vals else 1

                fraud_flags = [node_is_fraud[nb] for nb in neighbor_ids if nb in node_is_fraud]
                graph_neighbor_fraud_rate[node] = float(np.mean(fraud_flags)) if fraud_flags else 0.0
                graph_neighbor_fraud_count[node] = float(np.sum(fraud_flags)) if fraud_flags else 0.0

                risk_vals = [node_running_risk[nb] for nb in neighbor_ids if nb in node_running_risk]
                graph_avg_neighbor_risk[node] = float(np.mean(risk_vals)) if risk_vals else 0.0

                if ENABLE_EXPENSIVE_CENTRALITY:
                    bt_vals = [betweenness_cache[nb] for nb in neighbor_ids if nb in betweenness_cache]
                    graph_betweenness[node] = float(np.mean(bt_vals)) if bt_vals else 0.0
                    ev_vals = [eigenvector_cache[nb] for nb in neighbor_ids if nb in eigenvector_cache]
                    graph_eigenvector[node] = float(np.mean(ev_vals)) if ev_vals else 0.0

            graph_devices_per_card[node] = len(card_devices_seen.get(card, set())) or 1

        batch_nodes = batch["_node_id"].values
        G.add_nodes_from(batch_nodes)
        for _, group in batch.groupby(["card1", "DeviceInfo"]):
            nodes = group["_node_id"].tolist()
            if len(nodes) > 1:
                anchor = nodes[0]
                for other in nodes[1:]:
                    G.add_edge(anchor, other)
        for _, group in batch.groupby(["addr1", "DeviceInfo"]):
            nodes = group["_node_id"].tolist()
            if len(nodes) > 1:
                anchor = nodes[0]
                for other in nodes[1:]:
                    G.add_edge(anchor, other)
        for col in extra_edge_keys:
            for _, group in batch.groupby([col, "DeviceInfo"]):
                nodes = group["_node_id"].tolist()
                if len(nodes) > 1:
                    anchor = nodes[0]
                    for other in nodes[1:]:
                        G.add_edge(anchor, other)

        for card, dev, node in zip(batch["card1"].values, batch["DeviceInfo"].values, batch_nodes):
            card_device_index.setdefault((card, dev), []).append(node)
        for addr, dev, node in zip(batch["addr1"].values, batch["DeviceInfo"].values, batch_nodes):
            addr_device_index.setdefault((addr, dev), []).append(node)
        for col in extra_edge_keys:
            for val, dev, node in zip(batch[col].values, batch["DeviceInfo"].values, batch_nodes):
                extra_indexes[col].setdefault((val, dev), []).append(node)

        for node, label in zip(batch_nodes, batch["isFraud"].values):
            node_is_fraud[node] = int(label)

        for card, node, label in zip(batch["card1"].values, batch_nodes, batch["isFraud"].values):
            prior_fraud = card_fraud_count.get(card, 0)
            prior_count = card_txn_count.get(card, 0)
            node_running_risk[node] = (prior_fraud + BETA_PRIOR_ALPHA) / (
                prior_count + BETA_PRIOR_ALPHA + BETA_PRIOR_BETA
            )
            card_fraud_count[card] = prior_fraud + int(label)
            card_txn_count[card] = prior_count + 1

        for card, dev in zip(batch["card1"].values, batch["DeviceInfo"].values):
            card_devices_seen.setdefault(card, set()).add(dev)

        if batch_idx % 1 == 0:
            log(f"  Processed batch {batch_idx + 1}/{num_batches} "
                f"(graph now has {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges)")

    df["graph_degree"] = graph_degree
    df["graph_component_size"] = graph_component_size
    df["graph_clustering"] = graph_clustering
    df["graph_pagerank"] = graph_pagerank
    df["graph_community_size"] = graph_community_size
    df["graph_neighbor_fraud_rate"] = graph_neighbor_fraud_rate
    df["graph_neighbor_fraud_count"] = graph_neighbor_fraud_count
    df["graph_betweenness"] = graph_betweenness
    df["graph_eigenvector"] = graph_eigenvector
    df["graph_avg_neighbor_risk"] = graph_avg_neighbor_risk
    df["graph_devices_per_card"] = graph_devices_per_card

    df = df.drop(columns=["_node_id"])
    del G
    gc.collect()
    return df

## LAYER 3: DYNAMIC DEVICE TRUST SCORE 

In [12]:
DEVICE_COLS = [
    "device_txn_count", "device_unique_cards", "device_avg_gap",
    "device_trust_score", "device_recent_fraud_rate", "address_changed", "device_changed",
]

## Device Trust Features

In [13]:
def add_device_trust_features(df):
    log("Building Layer 3: dynamic device trust score (point-in-time, single-pass)...")

    df = df.sort_values("TransactionDT").reset_index(drop=True)
    n = len(df)

    device_txn_count = np.zeros(n, dtype=np.float32)
    device_unique_cards = np.ones(n, dtype=np.float32)
    device_avg_gap = np.full(n, -1, dtype=np.float32)
    device_trust_score = np.ones(n, dtype=np.float32)
    device_recent_fraud_rate = np.zeros(n, dtype=np.float32)

    device_count = {}
    device_cards_seen = {}
    device_last_time = {}
    device_gap_sum = {}
    device_gap_count = {}
    device_fraud_sum = {}
    device_recent_labels = {}
    global_prior_fraud_rate = BETA_PRIOR_ALPHA / (BETA_PRIOR_ALPHA + BETA_PRIOR_BETA)

    devices = df["DeviceInfo"].values
    cards = df["card1"].values
    times = df["TransactionDT"].values
    labels = df["isFraud"].values

    for i in range(n):
        dev = devices[i]
        card = cards[i]
        t = times[i]

        count_so_far = device_count.get(dev, 0)
        device_txn_count[i] = count_so_far
        device_unique_cards[i] = len(device_cards_seen.get(dev, set())) if dev in device_cards_seen else 1

        gap_count = device_gap_count.get(dev, 0)
        if gap_count > 0:
            device_avg_gap[i] = device_gap_sum[dev] / gap_count

        fraud_count = device_fraud_sum.get(dev, 0)
        posterior_fraud_rate = (fraud_count + BETA_PRIOR_ALPHA) / (count_so_far + BETA_PRIOR_ALPHA + BETA_PRIOR_BETA)
        device_trust_score[i] = 1 - posterior_fraud_rate

        recent = device_recent_labels.get(dev)
        device_recent_fraud_rate[i] = float(np.mean(recent)) if recent else global_prior_fraud_rate

        device_count[dev] = count_so_far + 1
        device_cards_seen.setdefault(dev, set()).add(card)
        if dev in device_last_time:
            gap = t - device_last_time[dev]
            device_gap_sum[dev] = device_gap_sum.get(dev, 0) + gap
            device_gap_count[dev] = gap_count + 1
        device_last_time[dev] = t
        device_fraud_sum[dev] = fraud_count + int(labels[i])
        device_recent_labels.setdefault(dev, deque(maxlen=DEVICE_RECENT_WINDOW)).append(int(labels[i]))

    df["device_txn_count"] = device_txn_count
    df["device_unique_cards"] = device_unique_cards
    df["device_avg_gap"] = device_avg_gap
    df["device_trust_score"] = device_trust_score
    df["device_recent_fraud_rate"] = device_recent_fraud_rate

    card_grouped = df.groupby("card1")
    df["address_changed"] = (
        card_grouped["addr1"].transform(lambda s: (s != s.shift(1)).astype(int))
    ).fillna(0)
    df["device_changed"] = (
        card_grouped["DeviceInfo"].transform(lambda s: (s != s.shift(1)).astype(int))
    ).fillna(0)

    return df

## PREPROCESSING 

In [14]:
def preprocess(df):
    log("Preprocessing: encoding categoricals and handling missing values...")
    y = df["isFraud"]
    X = df.drop(columns=["isFraud", "TransactionID"])

    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    encoders = {}
    for col in cat_cols:
        X[col] = X[col].fillna("missing").astype(str)
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
        encoders[col] = le

    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    X[num_cols] = X[num_cols].fillna(-999)
    for col in num_cols:
        if X[col].dtype == np.float64:
            X[col] = X[col].astype(np.float32)
        elif X[col].dtype == np.int64:
            X[col] = X[col].astype(np.int32)

    return X, y, encoders, cat_cols


In [15]:
def get_base_cols(X):
    engineered = set(TEMPORAL_COLS + GRAPH_COLS + DEVICE_COLS)
    return [c for c in X.columns if c not in engineered]

## FEATURE SELECTION

In [16]:
def select_reduced_features(X_train, y_train, base_cols, must_keep_cols,
                             drop_fraction=FEATURE_DROP_FRACTION):
    log("=" * 60)
    log(f"Selecting reduced feature set via SHAP importance "
        f"(dropping bottom {drop_fraction:.0%} of raw features)...")
    log("=" * 60)

    must_keep_cols = [c for c in dict.fromkeys(must_keep_cols) if c in X_train.columns]
    ranking_cols = [c for c in dict.fromkeys(base_cols + must_keep_cols) if c in X_train.columns]

    scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    ranker = quick_xgb(scale_pos_weight)
    ranker.fit(X_train[ranking_cols], y_train)

    sample_n = min(SHAP_RANKING_SAMPLE_SIZE, len(X_train))
    X_shap_sample = X_train[ranking_cols].sample(n=sample_n, random_state=RANDOM_STATE)
    explainer = shap.TreeExplainer(ranker)
    shap_values = explainer.shap_values(X_shap_sample)
    importances = pd.Series(np.abs(shap_values).mean(axis=0), index=ranking_cols)
    importances.sort_values(ascending=False).to_csv(
        os.path.join(OUTPUT_DIR, "feature_importance_shap_ranking.csv"), header=["mean_abs_shap"]
    )

    raw_only_cols = [c for c in base_cols if c in importances.index]
    ranked_raw = importances[raw_only_cols].sort_values(ascending=False)
    num_to_keep = int(np.ceil(len(ranked_raw) * (1 - drop_fraction)))
    selected_raw_cols = ranked_raw.head(num_to_keep).index.tolist()

    reduced_full_cols = selected_raw_cols + must_keep_cols

    log(f"  Kept {len(must_keep_cols)} protected engineered features "
        f"(temporal/graph/device-trust/anomaly/sequence) untouched.")
    log(f"  Kept top {len(selected_raw_cols)} of {len(raw_only_cols)} raw features by SHAP importance "
        f"(dropped bottom {len(raw_only_cols) - len(selected_raw_cols)}, ~{drop_fraction:.0%}).")
    log(f"  Reduced total feature count: {len(ranking_cols)} -> {len(reduced_full_cols)}.")

    del ranker, explainer, shap_values, X_shap_sample
    gc.collect()
    return selected_raw_cols, reduced_full_cols

##  SEQUENCE MODEL (LSTM) 

In [17]:
SEQ_FEATURE_COLS = [
    "TransactionAmt", "time_since_last_txn", "amt_drift", "amt_shock_ratio",
    "txn_count_so_far", "txn_velocity_24h", "device_switch_rate_5",
    "device_trust_score", "graph_neighbor_fraud_rate",
]

In [19]:
def build_sequences(df, cols, seq_len):
    log("Building per-card sequences for the LSTM (this can take a while)...")
    values = df[cols].values.astype(np.float32)
    card_ids = df["card1"].values
    n = len(df)
    num_features = len(cols)

    sequences = np.zeros((n, seq_len, num_features), dtype=np.float32)

    card_to_rows = {}
    for i, c in enumerate(card_ids):
        card_to_rows.setdefault(c, []).append(i)

    for card, rows in card_to_rows.items():
        history = []
        for i in rows:
            if history:
                past = history[-seq_len:]
                sequences[i, seq_len - len(past):, :] = np.array(past, dtype=np.float32)
            history.append(values[i])

    return sequences

In [20]:
EMBED_DIM = 8

##  Train Sequence Model

In [21]:
def train_sequence_model(train_idx, test_idx, X_all, y_all, cols, seq_len):
    log("Training sequence model (LSTM)...")
    try:
        import tensorflow as tf
        from tensorflow.keras.models import Model
        from tensorflow.keras.layers import LSTM, Dense, Masking, Input
    except ImportError:
        log("  TensorFlow not available - skipping sequence model. "
            "Install with: pip install tensorflow")
        n = len(X_all)
        return np.zeros(n, dtype=np.float32), np.zeros((n, EMBED_DIM), dtype=np.float32), None, None

    sequences = build_sequences(X_all, cols, seq_len)

    inputs = Input(shape=(seq_len, len(cols)))
    x = Masking(mask_value=0.0)(inputs)
    x = LSTM(16, use_cudnn=False)(x)
    embedding_layer = Dense(EMBED_DIM, activation="relu", name="embedding")(x)
    output_layer = Dense(1, activation="sigmoid")(embedding_layer)

    model = Model(inputs=inputs, outputs=output_layer)
    embedding_model = Model(inputs=inputs, outputs=embedding_layer)
    model.compile(optimizer="adam", loss="binary_crossentropy")

    X_seq_train = sequences[train_idx]
    y_seq_train = y_all.values[train_idx]

    model.fit(
        X_seq_train, y_seq_train,
        epochs=2, batch_size=1024, verbose=1,
        class_weight={0: 1.0, 1: (y_seq_train == 0).sum() / max((y_seq_train == 1).sum(), 1)},
    )

    train_probs = model.predict(sequences[train_idx], batch_size=4096, verbose=0).flatten()
    test_probs = model.predict(sequences[test_idx], batch_size=4096, verbose=0).flatten()
    train_embed = embedding_model.predict(sequences[train_idx], batch_size=4096, verbose=0)
    test_embed = embedding_model.predict(sequences[test_idx], batch_size=4096, verbose=0)

    n = len(X_all)
    seq_probs = np.zeros(n, dtype=np.float32)
    seq_probs[train_idx] = train_probs
    seq_probs[test_idx] = test_probs

    seq_embed = np.zeros((n, EMBED_DIM), dtype=np.float32)
    seq_embed[train_idx] = train_embed
    seq_embed[test_idx] = test_embed

    gc.collect()
    return seq_probs.astype(np.float32), seq_embed.astype(np.float32), model, embedding_model


## RISK TREND FEATURES 

In [22]:
RISK_TREND_COLS = ["previous_risk_mean_3", "previous_risk_max_5", "risk_change_rate"]

 ## how a card's OWN predicted risk is evolving 

In [23]:
def add_risk_trend_features(X):
    log("Building risk-trend features (evolving fraud pattern signal)...")
    grouped = X.groupby("card1")["sequence_model_prob"]

    X["previous_risk_mean_3"] = grouped.transform(
        lambda s: s.shift().rolling(window=3, min_periods=1).mean()
    ).fillna(0)
    X["previous_risk_max_5"] = grouped.transform(
        lambda s: s.shift().rolling(window=5, min_periods=1).max()
    ).fillna(0)

    short_risk = grouped.transform(lambda s: s.shift().rolling(window=3, min_periods=1).mean())
    long_risk = grouped.transform(lambda s: s.shift().rolling(window=10, min_periods=1).mean())
    X["risk_change_rate"] = (short_risk - long_risk).fillna(0)

    return X

## ANOMALY DETECTION (Isolation Forest)

In [24]:
def train_anomaly_detector(X_train, X_all, anomaly_cols):
    log("Training anomaly detector (Isolation Forest, on engineered features)...")
    iso = IsolationForest(
        n_estimators=150, contamination="auto", random_state=RANDOM_STATE, n_jobs=-1,
    )
    iso.fit(X_train[anomaly_cols])
    raw_scores = -iso.decision_function(X_all[anomaly_cols])
    scaled = (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min() + 1e-9)
    gc.collect()
    return scaled.astype(np.float32), iso


## Train Autoencoder Anomaly Detector

In [25]:
def train_autoencoder_anomaly_detector(X_train, X_all, anomaly_cols):
    log("Training autoencoder anomaly detector (additional channel)...")
    try:
        import tensorflow as tf
        from tensorflow.keras.models import Model
        from tensorflow.keras.layers import Input, Dense
    except ImportError:
        log("  TensorFlow not available - skipping autoencoder anomaly channel.")
        return np.zeros(len(X_all), dtype=np.float32), None, None

    from sklearn.preprocessing import StandardScaler as _Scaler
    scaler = _Scaler()
    X_train_scaled = scaler.fit_transform(X_train[anomaly_cols])
    X_all_scaled = scaler.transform(X_all[anomaly_cols])

    input_dim = len(anomaly_cols)
    bottleneck_dim = max(4, input_dim // 8)

    inputs = Input(shape=(input_dim,))
    encoded = Dense(32, activation="relu")(inputs)
    encoded = Dense(bottleneck_dim, activation="relu")(encoded)
    decoded = Dense(32, activation="relu")(encoded)
    decoded = Dense(input_dim, activation="linear")(decoded)

    autoencoder = Model(inputs, decoded)
    autoencoder.compile(optimizer="adam", loss="mse")
    autoencoder.fit(
        X_train_scaled, X_train_scaled,
        epochs=5, batch_size=1024, verbose=1, shuffle=True,
    )

    reconstructed = autoencoder.predict(X_all_scaled, batch_size=4096, verbose=0)
    reconstruction_error = np.mean((X_all_scaled - reconstructed) ** 2, axis=1)
    scaled_error = (reconstruction_error - reconstruction_error.min()) / (
        reconstruction_error.max() - reconstruction_error.min() + 1e-9
    )

    gc.collect()
    return scaled_error.astype(np.float32), autoencoder, scaler


## EARLY-WARNING LABEL 

In [26]:
def build_lookahead_label(df, k):
    log(f"Building early-warning label (fraud in next {k} transactions per card)...")
    grouped = df.groupby("card1")["isFraud"]
    reversed_fraud = grouped.transform(lambda s: s[::-1].shift(1).rolling(window=k, min_periods=1).sum()[::-1])
    return (reversed_fraud.fillna(0) > 0).astype(int)

## Metrics

In [27]:
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "auc": roc_auc_score(y_true, y_prob),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }

## Quick Xgb

In [28]:
def quick_xgb(scale_pos_weight, params=None):
    base_params = dict(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight, eval_metric="auc",
        tree_method="hist", device=XGB_DEVICE, random_state=RANDOM_STATE, n_jobs=-1,
    )
    if params:
        base_params.update(params)
    return xgb.XGBClassifier(**base_params)

## BASELINE COMPARISON (chronological split)

In [29]:
def run_baseline_comparison(X_train, X_test, y_train, y_test, base_cols, full_cols, tuned_params=None):
    log("=" * 60)
    log("Running baseline comparison (chronological split)...")
    log("=" * 60)

    scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    results = []

    log("  Training baseline: XGBoost (raw features only)...")
    xgb_raw = quick_xgb(scale_pos_weight)
    xgb_raw.fit(X_train[base_cols], y_train)
    y_prob = xgb_raw.predict_proba(X_test[base_cols])[:, 1]
    m = compute_metrics(y_test, y_prob)
    m["model"] = "XGBoost (raw features)"
    results.append(m)

    log("  Training baseline: Random Forest (raw features only)...")
    rf = RandomForestClassifier(
        n_estimators=200, max_depth=12, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    rf.fit(X_train[base_cols], y_train)
    y_prob = rf.predict_proba(X_test[base_cols])[:, 1]
    m = compute_metrics(y_test, y_prob)
    m["model"] = "Random Forest (raw features)"
    results.append(m)

    log("  Training baseline: Logistic Regression (raw features only)...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train[base_cols])
    X_test_scaled = scaler.transform(X_test[base_cols])
    lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
    lr.fit(X_train_scaled, y_train)
    y_prob = lr.predict_proba(X_test_scaled)[:, 1]
    m = compute_metrics(y_test, y_prob)
    m["model"] = "Logistic Regression (raw features)"
    results.append(m)

    del xgb_raw, rf, lr, scaler, X_train_scaled, X_test_scaled
    gc.collect()

    log("  Training proposed model: Full framework (dynamic graph + device trust + "
        "sequence + anomaly)...")
    proposed = quick_xgb(scale_pos_weight, params=tuned_params)
    proposed.fit(X_train[full_cols], y_train)
    y_prob = proposed.predict_proba(X_test[full_cols])[:, 1]
    m = compute_metrics(y_test, y_prob)
    m["model"] = "Proposed Framework (All Components)"
    results.append(m)

    results_df = pd.DataFrame(results)[["model", "auc", "precision", "recall", "f1", "mcc"]]
    results_df.to_csv(os.path.join(OUTPUT_DIR, "baseline_comparison.csv"), index=False)
    log("\n" + results_df.to_string(index=False))
    return results_df, proposed

## THRESHOLD OPTIMIZATION

In [30]:
def optimize_threshold(y_test, y_prob, beta=2.0, min_precision=None):
    candidate_thresholds = np.round(np.arange(0.30, 0.901, 0.01), 2)
    beta_sq = beta ** 2

    records = []
    for t in candidate_thresholds:
        m = compute_metrics(y_test, y_prob, threshold=t)
        p, r = m["precision"], m["recall"]
        denom = beta_sq * p + r
        m["fbeta"] = (1 + beta_sq) * p * r / denom if denom > 0 else 0.0
        m["threshold"] = t
        records.append(m)

    results_df = pd.DataFrame(records)[["threshold", "precision", "recall", "f1", "fbeta", "mcc", "auc"]]
    results_df.to_csv(os.path.join(OUTPUT_DIR, "threshold_comparison.csv"), index=False)

    eligible_df = results_df[results_df["precision"] >= min_precision] if min_precision else results_df
    if eligible_df.empty:
        log(f"  WARNING: no threshold met min_precision={min_precision}; falling back to the full grid.")
        eligible_df = results_df

    best_row = eligible_df.loc[eligible_df["fbeta"].idxmax()]
    best_threshold = float(best_row["threshold"])
    optimized_metrics = compute_metrics(y_test, y_prob, threshold=best_threshold)

    focus_band = results_df[(results_df["threshold"] >= 0.55) & (results_df["threshold"] <= 0.70)]
    log("  Threshold comparison, 0.55-0.70 band (precision/recall trade-off around 0.60-0.65):")
    log("\n" + focus_band.to_string(index=False))
    log(f"  Selected threshold: {best_threshold:.2f} "
        f"(F{beta:.0f}-optimal{' subject to min_precision=' + str(min_precision) if min_precision else ''})")

    return best_threshold, optimized_metrics

## THRESHOLD OPTIMIZATION (PLOT CURVES)

In [31]:
def plot_curves(y_test, y_prob):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc_val = auc(fpr, tpr)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc_val:.3f})")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve - Proposed Model")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "roc_curve.png"), dpi=150)
    plt.close()

    precisions, recalls, _ = precision_recall_curve(y_test, y_prob)
    plt.figure(figsize=(6, 5))
    plt.plot(recalls, precisions)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve - Proposed Model")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "pr_curve.png"), dpi=150)
    plt.close()

## EXPLAINABILITY

In [32]:
def explain_model(model, X_test):
    log("Computing SHAP values (global + local explainability)...")
    explainer = shap.TreeExplainer(model)
    sample = X_test.sample(500,random_state=RANDOM_STATE)
    shap_values = explainer.shap_values(sample)

    plt.figure()
    shap.summary_plot(shap_values, sample, show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "shap_summary_plot.png"), dpi=150)
    plt.close()

    plt.figure()
    shap.force_plot(
        explainer.expected_value, shap_values[0, :], sample.iloc[0, :],
        matplotlib=True, show=False,
    )
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "shap_local_example.png"), dpi=150)
    plt.close()
    return explainer

##  Risk Evolution For Card PLOT

In [33]:
def plot_risk_evolution_for_card(model, explainer, X, df, full_cols, card1_value):
    log(f"Plotting risk evolution for card1={card1_value}...")
    card_rows = df.index[df["card1"] == card1_value].tolist()
    if len(card_rows) < 2:
        log("  Not enough transactions for this card to plot evolution - skipping.")
        return

    probs = model.predict_proba(X.loc[card_rows, full_cols])[:, 1]

    plt.figure(figsize=(8, 4))
    plt.plot(range(len(card_rows)), probs, marker="o")
    plt.xlabel("Transaction number (chronological, this card)")
    plt.ylabel("Predicted fraud probability")
    plt.title(f"Risk Evolution Over Time - card1={card1_value}")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "risk_evolution_example.png"), dpi=150)
    plt.close()

## Plot Feature Importance

In [35]:
def plot_feature_importance(model, cols):
    importance = model.feature_importances_
    feat_imp = pd.Series(importance, index=cols).sort_values(ascending=False).head(20)
    plt.figure(figsize=(8, 6))
    feat_imp.sort_values().plot(kind="barh")
    plt.title("Top 20 Feature Importances (Proposed Model)")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "feature_importance.png"), dpi=150)
    plt.close()

## DORMANCY RISK SCORING (project risk for currently-inactive/silent cards)

In [36]:
def score_dormant_accounts(model, df, X, full_cols, dormancy_days_to_check=(0, 1, 3, 7, 14, 30), top_n_cards=10):
    log("Scoring dormancy risk projections for inactive cards...")

    last_txn_idx = df.groupby("card1").tail(1).index
    candidate_cards = df.loc[last_txn_idx, "card1"]
    txn_counts = df["card1"].value_counts()
    candidate_cards = candidate_cards[candidate_cards.map(txn_counts) >= 3].head(top_n_cards)

    records = []
    for card in candidate_cards:
        row_idx = df.index[(df["card1"] == card)][-1]
        base_features = X.loc[row_idx, full_cols].copy()

        for days in dormancy_days_to_check:
            projected = base_features.copy()
            projected["time_since_last_txn"] = days * 86400
            if "txn_velocity_24h" in projected.index:
                projected["txn_velocity_24h"] = 0
            prob = model.predict_proba(projected.values.reshape(1, -1))[0, 1]
            records.append({"card1": card, "dormancy_days": days, "projected_fraud_risk": prob})

    result_df = pd.DataFrame(records)
    result_df.to_csv(os.path.join(OUTPUT_DIR, "dormancy_risk_projections.csv"), index=False)

    plt.figure(figsize=(8, 5))
    for card, group in result_df.groupby("card1"):
        plt.plot(group["dormancy_days"], group["projected_fraud_risk"], marker="o", label=f"card1={card}")
    plt.xlabel("Days of inactivity before next transaction")
    plt.ylabel("Projected fraud risk (if it transacts after this many days)")
    plt.title("Dormancy Risk Projection (example cards)")
    plt.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "dormancy_risk_projection.png"), dpi=150)
    plt.close()

    log("Dormancy risk projections saved -> dormancy_risk_projections.csv, dormancy_risk_projection.png")
    return result_df

## Run Hyperparameter Tuning

In [37]:
def run_hyperparameter_tuning(X_train, y_train, full_cols):
    log("=" * 60)
    log("Running hyperparameter tuning (RandomizedSearchCV)...")
    log("=" * 60)

    if len(X_train) > TUNING_SAMPLE_SIZE:
        tuning_idx = X_train.sample(n=TUNING_SAMPLE_SIZE, random_state=RANDOM_STATE).index
    else:
        tuning_idx = X_train.index

    X_tune = X_train.loc[tuning_idx, full_cols]
    y_tune = y_train.loc[tuning_idx]
    scale_pos_weight = (y_tune == 0).sum() / max((y_tune == 1).sum(), 1)

    param_grid = {
        "max_depth": [4, 6, 8, 10],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "n_estimators": [200, 300, 400, 600],
        "subsample": [0.6, 0.8, 1.0],
        "colsample_bytree": [0.6, 0.8, 1.0],
    }

    base_model = xgb.XGBClassifier(
        scale_pos_weight=scale_pos_weight, eval_metric="auc",
        tree_method="hist", device=XGB_DEVICE, random_state=RANDOM_STATE, n_jobs=-1,
    )
    search = RandomizedSearchCV(
        base_model, param_distributions=param_grid, n_iter=HYPERPARAM_SEARCH_ITER,
        scoring="roc_auc", cv=3, random_state=RANDOM_STATE, n_jobs=-1, verbose=2,
    )
    search.fit(X_tune, y_tune)

    log(f"  Best CV ROC-AUC during search: {search.best_score_:.4f}")
    log(f"  Best params: {search.best_params_}")
    with open(os.path.join(OUTPUT_DIR, "best_hyperparameters.json"), "w") as f:
        json.dump(search.best_params_, f, indent=2)
    return search.best_params_

## Ablation Study

In [38]:
def run_ablation_study(X_train, X_test, y_train, y_test, base_cols, full_cols, tuned_params):
    log("=" * 60)
    log("Running ablation study...")
    log("=" * 60)

    scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    configs = {
        "Raw only": base_cols,
        "Raw + Temporal": base_cols + TEMPORAL_COLS,
        "Raw + Temporal + Graph": base_cols + TEMPORAL_COLS + GRAPH_COLS,
        "Full Model (+ Device Trust + Sequence + Anomaly)": full_cols,
    }

    results = []
    for name, cols in configs.items():
        log(f"  Training: {name} ({len(cols)} features)...")
        model = quick_xgb(scale_pos_weight, params=tuned_params)
        model.fit(X_train[cols], y_train)
        y_prob = model.predict_proba(X_test[cols])[:, 1]
        m = compute_metrics(y_test, y_prob)
        m["config"] = name
        m["num_features"] = len(cols)
        results.append(m)
        del model
    gc.collect()

    results_df = pd.DataFrame(results)[["config", "num_features", "auc", "precision", "recall", "f1", "mcc"]]
    results_df.to_csv(os.path.join(OUTPUT_DIR, "ablation_study.csv"), index=False)
    log("\n" + results_df.to_string(index=False))
    return results_df

## Stratified Cv Evaluation

In [39]:
def run_stratified_cv_evaluation(X, y, full_cols, tuned_params, n_splits=CV_N_SPLITS):
    log("=" * 60)
    log(f"Running {n_splits}-fold Stratified CV (final-evaluation reliability check)...")
    log("=" * 60)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=False)
    fold_records = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(X[full_cols], y), start=1):
        y_fold_train, y_fold_val = y.iloc[train_idx], y.iloc[val_idx]
        scale_pos_weight = (y_fold_train == 0).sum() / max((y_fold_train == 1).sum(), 1)
        model = quick_xgb(scale_pos_weight, params=tuned_params)
        model.fit(X.iloc[train_idx][full_cols], y_fold_train)
        y_prob = model.predict_proba(X.iloc[val_idx][full_cols])[:, 1]
        m = compute_metrics(y_fold_val, y_prob)
        m["fold"] = fold
        fold_records.append(m)
        log(f"  Fold {fold}/{n_splits}: AUC={m['auc']:.4f} Precision={m['precision']:.4f} "
            f"Recall={m['recall']:.4f} F1={m['f1']:.4f} MCC={m['mcc']:.4f}")
        del model
    gc.collect()

    cv_df = pd.DataFrame(fold_records)[["fold", "auc", "precision", "recall", "f1", "mcc"]]
    summary = cv_df[["auc", "precision", "recall", "f1", "mcc"]].agg(["mean", "std"])

    cv_df.to_csv(os.path.join(OUTPUT_DIR, "cv_fold_metrics.csv"), index=False)
    summary.to_csv(os.path.join(OUTPUT_DIR, "cv_summary.csv"))
    log("\n" + summary.to_string())

    return cv_df, summary

## Train Early Warning Model

In [40]:
def train_early_warning_model(X_train, X_test, lookahead_train, lookahead_test, full_cols, tuned_params):
    log("=" * 60)
    log("Training EARLY-WARNING model (trained directly on the forward-looking label)...")
    log("=" * 60)

    scale_pos_weight = (lookahead_train == 0).sum() / max((lookahead_train == 1).sum(), 1)
    model = quick_xgb(scale_pos_weight, params=tuned_params)
    model.fit(X_train[full_cols], lookahead_train)

    y_prob = model.predict_proba(X_test[full_cols])[:, 1]
    metrics = compute_metrics(lookahead_test, y_prob) if lookahead_test.nunique() > 1 else {
        "auc": float("nan"), "precision": float("nan"), "recall": float("nan"),
        "f1": float("nan"), "mcc": float("nan"),
    }

    text = (
        f"EARLY-WARNING MODEL (trained on 'fraud in next {LOOKAHEAD_K} transactions', "
        f"not on same-transaction isFraud)\n"
        f"AUC={metrics['auc']:.4f}  Precision={metrics['precision']:.4f}  "
        f"Recall={metrics['recall']:.4f}  F1={metrics['f1']:.4f}  MCC={metrics['mcc']:.4f}\n"
    )
    with open(os.path.join(OUTPUT_DIR, "early_warning_model_metrics.txt"), "w") as f:
        f.write(text)
    log("\n" + text)

    model.save_model(os.path.join(OUTPUT_DIR, "model_early_warning.json"))
    return model, metrics

MAIN

In [ ]:
def main():
    df = load_data()
    df = add_temporal_features(df)
    df = add_dynamic_graph_features(df)
    df = add_device_trust_features(df)

    lookahead_label = build_lookahead_label(df, LOOKAHEAD_K)

    X, y, encoders, cat_cols = preprocess(df)
    base_cols = get_base_cols(X)

    split_idx = int(len(X) * 0.8)
    train_idx = np.arange(split_idx)
    test_idx = np.arange(split_idx, len(X))

    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
    lookahead_train = lookahead_label.iloc[:split_idx]
    lookahead_test = lookahead_label.iloc[split_idx:]

    seq_probs, seq_embed, sequence_model, embedding_model = train_sequence_model(
        train_idx, test_idx, X, y, SEQ_FEATURE_COLS, SEQUENCE_LENGTH
    )
    X["sequence_model_prob"] = seq_probs
    seq_embed_cols = [f"seq_embed_{i}" for i in range(EMBED_DIM)]
    for i, col in enumerate(seq_embed_cols):
        X[col] = seq_embed[:, i]

    X = add_risk_trend_features(X)

    engineered_cols_for_anomaly = base_cols + TEMPORAL_COLS + GRAPH_COLS + DEVICE_COLS
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    anomaly_scores, isolation_forest = train_anomaly_detector(X_train, X, engineered_cols_for_anomaly)
    X["anomaly_score"] = anomaly_scores

    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    autoencoder_scores, autoencoder_model, autoencoder_scaler = train_autoencoder_anomaly_detector(
        X_train, X, engineered_cols_for_anomaly
    )
    X["autoencoder_score"] = autoencoder_scores

    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    full_cols_all = (
        base_cols + TEMPORAL_COLS + GRAPH_COLS + DEVICE_COLS + RISK_TREND_COLS
        + ["sequence_model_prob", "anomaly_score", "autoencoder_score"] + seq_embed_cols
    )

    protected_cols = (
        TEMPORAL_COLS + GRAPH_COLS + DEVICE_COLS + RISK_TREND_COLS
        + ["sequence_model_prob", "anomaly_score", "autoencoder_score"] + seq_embed_cols
    )
    base_cols, full_cols = select_reduced_features(
        X_train, y_train, base_cols, must_keep_cols=protected_cols, drop_fraction=FEATURE_DROP_FRACTION
    )

    tuned_params = run_hyperparameter_tuning(X_train, y_train, full_cols)

    run_ablation_study(X_train, X_test, y_train, y_test, base_cols, full_cols, tuned_params)

    baseline_df, proposed_model = run_baseline_comparison(X_train, X_test, y_train, y_test, base_cols, full_cols, tuned_params)

    run_stratified_cv_evaluation(X, y, full_cols, tuned_params, n_splits=CV_N_SPLITS)

    y_prob = proposed_model.predict_proba(X_test[full_cols])[:, 1]
    best_threshold, optimized_metrics = optimize_threshold(y_test, y_prob)

    y_pred_opt = (y_prob >= best_threshold).astype(int)
    standard_text = (
        f"MODEL 1 - REAL-TIME TRANSACTION CLASSIFIER (trained on isFraud)\n"
        f"Optimized threshold: {best_threshold:.4f}\n"
        f"AUC={optimized_metrics['auc']:.4f}  Precision={optimized_metrics['precision']:.4f}  "
        f"Recall={optimized_metrics['recall']:.4f}  F1={optimized_metrics['f1']:.4f}  "
        f"MCC={optimized_metrics['mcc']:.4f}\n\n"
        f"Confusion matrix:\n{confusion_matrix(y_test, y_pred_opt)}\n\n"
        f"{classification_report(y_test, y_pred_opt)}\n"
    )

    early_auc = roc_auc_score(lookahead_test, y_prob) if lookahead_test.nunique() > 1 else float("nan")
    early_text = (
        f"MODEL 1's early-warning correlation check (probability vs whether "
        f"fraud occurs in the NEXT {LOOKAHEAD_K} transactions): AUC={early_auc:.4f}\n"
    )

    full_text = standard_text + "\n" + early_text
    with open(os.path.join(OUTPUT_DIR, "metrics_final_model.txt"), "w") as f:
        f.write(full_text)
    log("\n" + full_text)

    early_warning_model, early_warning_metrics = train_early_warning_model(
        X_train, X_test, lookahead_train, lookahead_test, full_cols, tuned_params
    )

    plot_curves(y_test, y_prob)
    plot_feature_importance(proposed_model, full_cols)
    explainer = explain_model(proposed_model, X_test[full_cols])

    example_card = df["card1"].value_counts()
    example_card = example_card[example_card > 1].index[0] if (example_card > 1).any() else None
    if example_card is not None:
        plot_risk_evolution_for_card(proposed_model, explainer, X, df, full_cols, example_card)

    score_dormant_accounts(proposed_model, df, X, full_cols)

    proposed_model.save_model(os.path.join(OUTPUT_DIR, "model_xgboost.json"))
    log(f"ALL DONE. Every result is saved in: {os.path.abspath(OUTPUT_DIR)}")

   
    import joblib

    MODEL_DIR = "models"
    os.makedirs(MODEL_DIR, exist_ok=True)
    log(f"Exporting production inference artifacts to {os.path.abspath(MODEL_DIR)} ...")

    proposed_model.save_model(os.path.join(MODEL_DIR, "model_xgboost.json"))

    encoders_serializable = {
        col: {str(cls): int(idx) for idx, cls in enumerate(le.classes_)}
        for col, le in encoders.items()
    }
    with open(os.path.join(MODEL_DIR, "feature_encoders.json"), "w") as f:
        json.dump(encoders_serializable, f)

    with open(os.path.join(MODEL_DIR, "cat_cols.json"), "w") as f:
        json.dump(cat_cols, f)

    with open(os.path.join(MODEL_DIR, "feature_list.json"), "w") as f:
        json.dump(full_cols, f)

    with open(os.path.join(MODEL_DIR, "threshold.json"), "w") as f:
        json.dump({"threshold": float(best_threshold)}, f)

    with open(os.path.join(MODEL_DIR, "anomaly_cols.json"), "w") as f:
        json.dump(engineered_cols_for_anomaly, f)

    with open(os.path.join(MODEL_DIR, "seq_feature_cols.json"), "w") as f:
        json.dump(SEQ_FEATURE_COLS, f)

    with open(os.path.join(MODEL_DIR, "seq_config.json"), "w") as f:
        json.dump({"seq_len": SEQUENCE_LENGTH, "embed_dim": EMBED_DIM}, f)

    if sequence_model is not None:
        sequence_model.save(os.path.join(MODEL_DIR, "lstm_sequence_model.keras"))
    else:
        log("  WARNING: sequence model was not trained (TensorFlow unavailable) - "
            "'sequence_model_prob'/'seq_embed_*' will default to 0 at inference.")

    if autoencoder_model is not None:
        autoencoder_model.save(os.path.join(MODEL_DIR, "autoencoder_model.keras"))
        joblib.dump(autoencoder_scaler, os.path.join(MODEL_DIR, "autoencoder_scaler.pkl"))
    else:
        log("  WARNING: autoencoder was not trained (TensorFlow unavailable) - "
            "'autoencoder_score' will default to 0 at inference.")

    joblib.dump(isolation_forest, os.path.join(MODEL_DIR, "isolation_forest.pkl"))

    log(f"  Saved: model_xgboost.json, feature_encoders.json, cat_cols.json, "
        f"feature_list.json ({len(full_cols)} features), threshold.json, "
        f"anomaly_cols.json, seq_feature_cols.json, seq_config.json, "
        f"lstm_sequence_model.keras, autoencoder_model.keras + scaler, isolation_forest.pkl")
    log("Production artifact export complete.")


## Script Entry Point

In [42]:
if __name__ == "__main__":
    main()

[14:21:17] Loading transaction and identity tables...
[14:21:31] Loaded 590,540 rows and 434 columns, sorted chronologically.
[14:21:31] Building Layer 1: expanded temporal features (point-in-time)...
[14:22:31] Building Layer 2: dynamic social graph features (batched)...
[14:22:49]   Processed batch 1/12 (graph now has 50,000 nodes, 63,552 edges)
[14:24:45]   Processed batch 2/12 (graph now has 100,000 nodes, 162,343 edges)
[14:28:32]   Processed batch 3/12 (graph now has 150,000 nodes, 233,518 edges)
[14:31:41]   Processed batch 4/12 (graph now has 200,000 nodes, 256,507 edges)
[14:35:10]   Processed batch 5/12 (graph now has 250,000 nodes, 279,525 edges)
[14:39:20]   Processed batch 6/12 (graph now has 300,000 nodes, 310,409 edges)
[14:43:09]   Processed batch 7/12 (graph now has 350,000 nodes, 330,803 edges)
[14:48:06]   Processed batch 8/12 (graph now has 400,000 nodes, 356,705 edges)
[14:53:01]   Processed batch 9/12 (graph now has 450,000 nodes, 381,591 edges)
[14:58:42]   Proce